## Kraken and Coinbase security helper

In [71]:
!pip install coinbase-advanced-py requests pandas numpy flask python-dotenv

- **coinbase-advanced-py** official Coinbase SDK
- **requests** for Kraken (no official Python SDK, but their REST API is clean)
- **pandas/numpy** risk score calculations
- **flask** lightweight API layer if you want a frontend to hit
- **python-dotenv** keep API keys out of your code

---

## Project Structure
```
/security-platform
.env                  # API keys
main.py               # entry point
coinbase_client.py    # Coinbase data fetching
kraken_client.py      # Kraken data fetching
risk_engine.py        # risk score logic
api.py                # Flask endpoints (optional)
```

---

## .env
```
COINBASE_API_KEY=your_key
COINBASE_API_SECRET=your_secret
KRAKEN_API_KEY=your_key
KRAKEN_API_SECRET=your_secret

## Coinbase

In [72]:
from datetime import datetime, timedelta

# Mock account info
COINBASE_USER = {
    "name": "Mike Johnson",
    "email": "mike.johnson@email.com",
    "username": "mikej_crypto",
    "created_at": "2021-03-15",
    "country": "US",
    "verified": True
}

def get_coinbase_profile():
    return COINBASE_USER

def get_coinbase_balances():
    return [
        {"asset": "BTC", "balance": 0.85, "account_id": "mock-btc-001"},
        {"asset": "ETH", "balance": 4.20, "account_id": "mock-eth-002"},
        {"asset": "USD", "balance": 3200.00, "account_id": "mock-usd-003"}
    ]

def get_coinbase_transactions(account_id, limit=100):
    now = datetime.utcnow()
    return [
        {"type": "buy",  "amount": 500.00,  "currency": "USD", "timestamp": (now - timedelta(days=30)).isoformat(), "status": "completed", "to": None,           "description": "Bought BTC"},
        {"type": "buy",  "amount": 300.00,  "currency": "USD", "timestamp": (now - timedelta(days=25)).isoformat(), "status": "completed", "to": None,           "description": "Bought ETH"},
        {"type": "send", "amount": 200.00,  "currency": "USD", "timestamp": (now - timedelta(days=20)).isoformat(), "status": "completed", "to": "addr-known-001", "description": "Sent to known wallet"},
        {"type": "send", "amount": 150.00,  "currency": "USD", "timestamp": (now - timedelta(days=15)).isoformat(), "status": "completed", "to": "addr-known-001", "description": "Sent to known wallet"},
        {"type": "send", "amount": 1200.00, "currency": "USD", "timestamp": (now - timedelta(days=5)).isoformat(),  "status": "completed", "to": "addr-new-001",   "description": "⚠️ Sent to NEW address"},
        {"type": "send", "amount": 950.00,  "currency": "USD", "timestamp": (now - timedelta(days=4)).isoformat(),  "status": "completed", "to": "addr-new-002",   "description": "⚠️ Sent to NEW address"},
        {"type": "send", "amount": 800.00,  "currency": "USD", "timestamp": (now - timedelta(days=3)).isoformat(),  "status": "completed", "to": "addr-new-003",   "description": "⚠️ Sent to NEW address"},
        {"type": "send", "amount": 1100.00, "currency": "USD", "timestamp": (now - timedelta(days=2)).isoformat(),  "status": "completed", "to": "addr-new-004",   "description": "⚠️ Sent to NEW address"},
        {"type": "send", "amount": 750.00,  "currency": "USD", "timestamp": (now - timedelta(days=1)).isoformat(),  "status": "completed", "to": "addr-new-005",   "description": "⚠️ Sent to NEW address"},
    ]

## Crypto converter (coingecko.com)

In [73]:
import requests

def get_crypto_prices():
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {
        "ids": "bitcoin,ethereum",
        "vs_currencies": "usd"
    }
    r = requests.get(url, params=params)
    data = r.json()
    return {
        "BTC": data["bitcoin"]["usd"],
        "ETH": data["ethereum"]["usd"],
        "XBT": data["bitcoin"]["usd"],  # Kraken calls BTC "XBT"
        "USD": 1.00,
        "ZUSD": 1.00                     # Kraken calls USD "ZUSD"
    }

prices = get_crypto_prices()
print("Live Prices:")
for asset, price in prices.items():
    if asset not in ["USD", "ZUSD", "XBT"]:
        print(f"  {asset}: ${price:,.2f}")

Live Prices:
  BTC: $70,523.00
  ETH: $2,151.36


## Kraken

In [74]:
import requests, hmac, hashlib, base64, urllib.parse, time, os
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("KRAKEN_API_KEY")
API_SECRET = os.getenv("KRAKEN_API_SECRET")
BASE_URL = "https://api.kraken.com"

def _sign(urlpath, data):
    postdata = urllib.parse.urlencode(data)
    encoded = (str(data['nonce']) + postdata).encode()
    message = urlpath.encode() + hashlib.sha256(encoded).digest()
    mac = hmac.new(base64.b64decode(API_SECRET), message, hashlib.sha512)
    return base64.b64encode(mac.digest()).decode()

def _private_request(endpoint, params={}):
    urlpath = f"/0/private/{endpoint}"
    data = {"nonce": str(int(time.time() * 1000)), **params}
    headers = {
        "API-Key": API_KEY,
        "API-Sign": _sign(urlpath, data)
    }
    r = requests.post(BASE_URL + urlpath, headers=headers, data=data)
    return r.json()['result']

def get_kraken_balances():
    result = _private_request("BalanceEx")
    balances = []
    for asset, data in result.items():
        bal = float(data['balance'])
        if bal > 0:
            balances.append({
                "asset": asset,
                "balance": bal,
                "hold": float(data.get('hold_trade', 0))
            })
    return balances

def get_kraken_transactions(start=None):
    params = {"start": start} if start else {}
    result = _private_request("Ledgers", params)
    ledger = result.get('ledger', {})
    txns = []
    for txid, entry in ledger.items():
        txns.append({
            "txid": txid,
            "type": entry['type'],
            "asset": entry['asset'],
            "amount": float(entry['amount']),
            "timestamp": entry['time'],
            "fee": float(entry['fee'])
        })
    return txns

## risk_engine.py

In [75]:
def get_kraken_balances():
    return [
        {"asset": "XBT", "balance": 0.32, "hold": 0.0},
        {"asset": "ETH", "balance": 2.10, "hold": 0.0},
        {"asset": "ZUSD", "balance": 1500.00, "hold": 0.0}
    ]

def get_kraken_transactions():
    from datetime import datetime, timedelta
    now = datetime.utcnow()
    return [
        {"txid": "t001", "type": "deposit", "asset": "ZUSD", "amount": 1500.00, "timestamp": (now - timedelta(days=30)).timestamp(), "fee": 0.0},
        {"txid": "t002", "type": "trade", "asset": "XBT", "amount": 0.32, "timestamp": (now - timedelta(days=28)).timestamp(), "fee": 0.5},
        {"txid": "t003", "type": "withdrawal", "asset": "ZUSD", "amount": 200.00, "timestamp": (now - timedelta(days=10)).timestamp(), "fee": 0.0},
        {"txid": "t004", "type": "withdrawal", "asset": "ZUSD", "amount": 180.00, "timestamp": (now - timedelta(days=3)).timestamp(), "fee": 0.0},
    ]

## main.py — Wire It All Together

In [77]:
def run_assessment():
    # Account Info
    profile = get_coinbase_profile()
    print(f"{'='*40}")
    print(f"  ACCOUNT INFO")
    print(f"{'='*40}")
    print(f"  Name:       {profile['name']}")
    print(f"  Username:   @{profile['username']}")
    print(f"  Email:      {profile['email']}")
    print(f"  Member Since: {profile['created_at']}")
    print(f"  Verified:   {'✅ Yes' if profile['verified'] else '❌ No'}")

    # Balances
    #print(f"\n{'='*40}")
    #print(f"  ACCOUNT BALANCES")
    #print(f"{'='*40}")
    #cb_balances = get_coinbase_balances()
    #for b in cb_balances:
    #    print(f"  {b['asset']:<6} {b['balance']:>10,.4f}")
    prices = get_crypto_prices()
    print(f"\n{'='*55}")
    print(f"  ACCOUNT BALANCES (LIVE USD VALUE)")
    print(f"{'='*55}")
    cb_balances = get_coinbase_balances()
    total_usd = 0
    print(f"  {'Asset':<6} {'Holdings':>12}  {'Price':>12}  {'USD Value':>12}")
    print(f"  {'-'*50}")
    for b in cb_balances:
        price = prices.get(b['asset'], 0)
        usd_value = b['balance'] * price
        total_usd += usd_value
        print(f"  {b['asset']:<6} {b['balance']:>12,.4f}  ${price:>11,.2f}  ${usd_value:>11,.2f}")
    print(f"  {'-'*50}")
    print(f"  {'LIQUIDATION VALUE':<32}  ${total_usd:>11,.2f}")

    # Transactions
    print(f"\n{'='*40}")
    print(f"  TRANSACTION HISTORY")
    print(f"{'='*40}")
    cb_txns = get_coinbase_transactions(cb_balances[0]['account_id'])
    print(f"  {'Date':<12} {'Type':<6} {'Amount':>10}  {'Description'}")
    print(f"  {'-'*55}")
    for t in cb_txns:
        date = t['timestamp'][:10]
        print(f"  {date:<12} {t['type']:<6} ${t['amount']:>8,.2f}  {t['description']}")

    # Risk Score
    kr_balances = get_kraken_balances()
    kr_txns = get_kraken_transactions()
    all_balances = cb_balances + kr_balances
    all_txns = cb_txns + kr_txns
    result = calculate_risk_score(all_txns, all_balances)

    print(f"\n{'='*40}")
    print(f"  RISK SCORE: {result['score']}/100 — {result['risk_level']}")
    print(f"{'='*40}")
    for flag in result['flags']:
        print(f"  ⚠️  {flag}")
    print(f"\n  Total balance (est): ${result['total_balance_usd_est']:,.2f}")
    print(f"  Transactions (7d):   {result['txn_count_7d']}")

run_assessment()

  ACCOUNT INFO
  Name:       Mike Johnson
  Username:   @mikej_crypto
  Email:      mike.johnson@email.com
  Member Since: 2021-03-15
  Verified:   ✅ Yes

  ACCOUNT BALANCES (LIVE USD VALUE)
  Asset      Holdings         Price     USD Value
  --------------------------------------------------
  BTC          0.8500  $  70,514.00  $  59,936.90
  ETH          4.2000  $   2,150.67  $   9,032.81
  USD      3,200.0000  $       1.00  $   3,200.00
  --------------------------------------------------
  LIQUIDATION VALUE                 $  72,169.71

  TRANSACTION HISTORY
  Date         Type       Amount  Description
  -------------------------------------------------------
  2026-02-19   buy    $  500.00  Bought BTC
  2026-02-24   buy    $  300.00  Bought ETH
  2026-03-01   send   $  200.00  Sent to known wallet
  2026-03-06   send   $  150.00  Sent to known wallet
  2026-03-16   send   $1,200.00  ⚠️ Sent to NEW address
  2026-03-17   send   $  950.00  ⚠️ Sent to NEW address
  2026-03-18   send

/var/folders/4j/v3j60w3j1mjgfl4x7tttdrhm0000gn/T/ipykernel_61002/3147012779.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/var/folders/4j/v3j60w3j1mjgfl4x7tttdrhm0000gn/T/ipykernel_61002/391613262.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
/var/folders/4j/v3j60w3j1mjgfl4x7tttdrhm0000gn/T/ipykernel_61002/414317635.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
